# Student Workbook: Python + Excel darba uzdevumi (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/blob/main/notebooks/student_workbook_latvia_excel_datasets_colab.ipynb)

Šī ir studentu versija kursa tēmai **"Excel dokumentu apstrāde ar Python"**.  
Tā ir veidota kā praktiska darba piezīmju grāmata ar vadītiem uzdevumiem, bet bez gataviem pilniem risinājumiem.

Šī ir **Google Colab draudzīga** versija: ja mape `latvia_gov_excel_datasets` nav pieejama lokāli, piezīmju grāmata automātiski lejupielādēs publiskos Excel failus no repozitorija `ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python`.


## Tematiskie bloki

1. `1_budget_reports` — vairāku Excel failu apvienošana  
2. `2_procurement_data` — iepirkumu datu tīrīšana  
3. `3_employee_registry` — filtrēšana, grupēšana un atskaite  
4. `4_municipality_data` — datu savienošana ar `merge`  
5. `5_mixed_quality_data` — haotiska Excel eksporta tīrīšana  
6. `6_reporting_template` — rezultātu ierakstīšana šablonā  
7. `7_chart_data` — diagrammas ievietošana ar `openpyxl`

## Darba princips

Katrā sadaļā:
- īss uzdevuma apraksts,
- ieteiktie soļi,
- tukšas vai daļēji sagatavotas koda šūnas,
- vieta jūsu pašu risinājumam.

Mērķis nav tikai panākt rezultātu, bet arī saprast:
- kāpēc tiek izmantots `pandas`,
- kad ir noderīgs `openpyxl`,
- kā automatizācija aizstāj manuālu darbu Excel vidē.

## Texta - Markdown šūnas

### Pamatdoma

Teksta jeb Markdown šūnas dod mums iespēju izmantojot minimālu sintaksti aprakstīt uzdevumus, piezīmes, jebko kas domāts cilvēkam.

### Sintakse

* Nesakārtots saraksts tiek rakstits ar zvaigznīti
- var lietota arī dash
* Sakārtots saraksts tiek rakstīts ar 1. un 2.

#### Sakārtota saraksta piemērs

1. Rīts
2. Diena
3. Pēcpusdiena

### Saites 🔗

Saites (links) izveido ar kvadrātiekavām un tad parastajām

* [Kursa Repozitorijs](https://github.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python)
* [Github Pamācība Markdown](https://docs.github.com/en/get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax)

### Formatējums

* Varam izcelt ar **kaut kas svarīgs** ar divām zvaigznēm apkārt izceļamajam
* Varam arī izcel *samēra svarīgs* ar vienu zvaigzni -> slīpsvītra
* Varam izmantot jebkādus Unicode simbolus kā piemēram ☀️ utt

### Attēli

Attēlus ievieto līdzīgi saitēm tikai pieliek priekša ! izsaukuma zīmi pirms kvadrāt iekavām

![RTU](https://www.rtu.lv/images/logo_en.svg?v=1.1)

Piezīme, veidojot saiti uz attēlu skatamies vai tas ir brīvi pieejams.

## 0. Sagatavošana

Šī versija paredzēta darbam gan lokāli, gan Google Colab vidē.

Datu avots:
- repozitorijs: `https://github.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python`
- datu mape: `https://github.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/tree/main/latvia_gov_excel_datasets`
- ģenerēšanas skripts: `scripts/create_excel_spreadsheets.py`

Nākamā koda šūna pārbaudīs bibliotēkas un, ja vajadzēs, lejupielādēs trūkstošos Excel failus no publiskā GitHub repozitorija.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import re
from urllib.request import urlretrieve

IN_COLAB = "google.colab" in sys.modules

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages and IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
elif missing_packages:
    print("Missing local packages:", ", ".join(missing_packages))
    print("If you are not in Colab, install them manually before running the rest of the notebook.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.chart import BarChart, Reference

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


In [ ]:
REPO_OWNER = "ValRCS"
REPO_NAME = "RTU_Digitalas_Prasmes_Excel_VBA_Python"
REPO_BRANCH = "main"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}"
RAW_DATASET_BASE_URL = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{REPO_BRANCH}/latvia_gov_excel_datasets"

DATA_ROOT = Path("latvia_gov_excel_datasets")
OUTPUT_DIR = Path("student_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILES = [
    "README.txt",
    "1_budget_reports/budget_2025-01.xlsx",
    "1_budget_reports/budget_2025-02.xlsx",
    "1_budget_reports/budget_2025-03.xlsx",
    "1_budget_reports/budget_2025-04.xlsx",
    "1_budget_reports/budget_2025-05.xlsx",
    "1_budget_reports/budget_2025-06.xlsx",
    "2_procurement_data/procurement_raw.xlsx",
    "3_employee_registry/employees.xlsx",
    "4_municipality_data/population.xlsx",
    "4_municipality_data/budget.xlsx",
    "5_mixed_quality_data/mixed_data.xlsx",
    "6_reporting_template/report_template.xlsx",
    "7_chart_data/monthly_summary.xlsx",
]

def ensure_public_datasets(data_root: Path = DATA_ROOT):
    missing_files = [
        relative_path
        for relative_path in DATASET_FILES
        if not (data_root / relative_path).exists()
    ]

    for relative_path in missing_files:
        local_path = data_root / relative_path
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_DATASET_BASE_URL}/{relative_path}"
        urlretrieve(url, local_path)

    return missing_files

downloaded_files = ensure_public_datasets()

if downloaded_files:
    print(f"Downloaded {len(downloaded_files)} dataset files from {REPO_URL}.")
else:
    print("All dataset files are already available locally.")

print("Running in Colab:", IN_COLAB)
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 1. Palīgfunkcijas

Šeit uzrakstiet dažas funkcijas, kas palīdzēs atkārtoti izmantot vienu un to pašu loģiku.

### Ieteikumi
Izveidojiet funkcijas:
- `parse_mixed_date(series)` — datumu pārveidošanai
- `parse_amount(value)` — summu pārveidošanai uz `float`
- `normalize_supplier_name(name)` — piegādātāju nosaukumu vienādošanai

Var sākt ar ļoti vienkāršu versiju un pēc tam uzlabot.

In [ ]:
def parse_mixed_date(series: pd.Series) -> pd.Series:
    # TODO: pārveidojiet dažādus datumu formātus uz datetime
    pass

def parse_amount(value):
    # TODO: pārveidojiet summas uz float, arī ja tās ir tekstā
    pass

def normalize_supplier_name(name: str) -> str:
    # TODO: izveidojiet vienkāršu piegādātāju nosaukumu normalizāciju
    pass

# 2. Budget reports — vairāku Excel failu apvienošana

## Uzdevums
Mapē `1_budget_reports` atrodas vairāki Excel faili ar budžeta datiem par dažādiem mēnešiem.

### Jums jāizdara:
1. Jāatrod visi `.xlsx` faili mapē.
2. Jāielasa tie ar `pandas`.
3. Jāpievieno kolonna ar avota faila nosaukumu.
4. Jāapvieno visi faili vienā DataFrame.
5. Jānormalizē summas un datumi.
6. Jāizveido kopsavilkums pa mēnešiem un nodaļām.
7. Jāeksportē rezultāts jaunā Excel failā.

### Padoms
Izmantojiet:
- `Path.glob()`
- `pd.read_excel()`
- `pd.concat()`
- `groupby()`

In [ ]:
budget_dir = DATA_ROOT / "1_budget_reports"

# TODO: iegūstiet budžeta failu sarakstu
budget_files = []

budget_files

In [ ]:
# TODO: nolasiet visus failus un apvienojiet vienā DataFrame
budget_frames = []

# piemērs idejai:
# for file in budget_files:
#     df = ...
#     df["Source File"] = ...
#     budget_frames.append(df)

budget_raw = None
budget_raw

In [ ]:
# TODO: iztīriet budžeta datus
# Ieteikumi:
# - pārveidojiet Amount EUR uz skaitli
# - pārveidojiet Date uz datetime
# - izveidojiet Month kolonnu
# - aizpildiet trūkstošās kategorijas

budget_clean = None
budget_clean.head()

In [ ]:
# TODO: izveidojiet kopsavilkumu pa mēnešiem un nodaļām
budget_summary_department = None

budget_summary_department

In [ ]:
# TODO: eksportējiet rezultātus uz Excel failu mapē OUTPUT_DIR
# Piemēram:
# solution_budget_reports.xlsx

# 3. Procurement data — iepirkumu datu tīrīšana

## Uzdevums
Failā `2_procurement_data/procurement_raw.xlsx` ir iepirkumu dati ar nekonsekventiem piegādātāju nosaukumiem.

### Jums jāizdara:
1. Jāielasa Excel fails.
2. Jāiztīra summas (`Amount EUR`).
3. Jānormalizē datumi.
4. Jānormalizē piegādātāju nosaukumi.
5. Jāizveido kopsavilkums pa piegādātājiem.
6. Jāeksportē rezultāts.

In [ ]:
proc_file = DATA_ROOT / "2_procurement_data" / "procurement_raw.xlsx"

# TODO: ielasiet failu
proc_raw = None
proc_raw.head()

In [ ]:
# TODO: izveidojiet iztīrīto versiju
# Ieteikumi:
# - Amount Clean
# - Date Parsed
# - Supplier Normalized

proc_clean = None
proc_clean.head()

In [ ]:
# TODO: izveidojiet kopsavilkumu
# Piemēram:
# - cik līgumu katram piegādātājam
# - kopējā summa
# - vidējā summa

supplier_summary = None
supplier_summary

In [ ]:
# TODO: eksportējiet iztīrītos iepirkumu datus uz Excel

# 4. Employee registry — filtrēšana un atskaite

## Uzdevums
Failā `3_employee_registry/employees.xlsx` ir darbinieku saraksts.

### Jums jāizdara:
1. Jāielasa dati.
2. Jāatrod darbinieki ar lielu algu.
3. Jāaprēķina vidējā alga pa nodaļām.
4. Jāizveido atskaite pa nodaļām.
5. Jāeksportē rezultāti.

In [ ]:
emp_file = DATA_ROOT / "3_employee_registry" / "employees.xlsx"

# TODO: ielasiet failu
emp = None
emp.head()

In [ ]:
# TODO: atlasiet, piemēram, darbiniekus ar Salary > 2200
high_salary = None
high_salary

In [ ]:
# TODO: izveidojiet nodaļu atskaiti
# Ieteikumi:
# - darbinieku skaits
# - vidējā alga
# - max/min alga
# - kopējais FTE

hr_report = None
hr_report

In [ ]:
# TODO: eksportējiet rezultātus uz Excel

# 5. Municipality data — datu savienošana ar `merge`

## Uzdevums
Jums ir divi faili:
- `population.xlsx`
- `budget.xlsx`

### Jums jāizdara:
1. Jāielasa abi faili.
2. Jāsavieno tie pēc `Municipality` un `Reference Year`.
3. Jāaprēķina:
   - budžets uz vienu iedzīvotāju,
   - kapitālie izdevumi uz vienu iedzīvotāju.
4. Jāsakārto rezultāts dilstošā secībā.
5. Jāizveido vienkāršs grafiks.

In [ ]:
population_file = DATA_ROOT / "4_municipality_data" / "population.xlsx"
budget_file = DATA_ROOT / "4_municipality_data" / "budget.xlsx"

# TODO: ielasiet abus failus
population = None
budget_m = None

In [ ]:
# TODO: savienojiet tabulas ar merge
municipality = None
municipality

In [ ]:
# TODO: aprēķiniet Budget per Capita un Capital per Capita
municipality_sorted = None
municipality_sorted.head(10)

In [ ]:
# TODO: uzzīmējiet top 10 pašvaldību stabiņu grafiku ar matplotlib

In [ ]:
# TODO: eksportējiet rezultātus uz Excel

# 6. Mixed quality data — haotiska Excel eksporta tīrīšana

## Uzdevums
Failā `5_mixed_quality_data/mixed_data.xlsx` dati nav glītā tabulas formā.

### Faila problēmas
- papildus virsraksta rindas,
- vairāk nekā viena galvene,
- tukšas rindas,
- dažādi kolonnu nosaukumi,
- summas kā teksts,
- datumu nekonsekvence,
- iespējamie dublikāti.

### Jums jāizdara
1. Jāielasa fails bez automātiskas galvenes.
2. Jāatrod galvenes rindu pozīcijas.
3. Jāizgriež datu bloki.
4. Jāapvieno tie vienā DataFrame.
5. Jānormalizē kolonnas un vērtības.
6. Jāizņem dublikāti.
7. Jāizveido kopsavilkums.

In [ ]:
mixed_file = DATA_ROOT / "5_mixed_quality_data" / "mixed_data.xlsx"

# TODO: ielasiet failu ar header=None
mixed_raw = None
mixed_raw.head(15)

In [ ]:
# TODO: atrodiet rindas, kur atrodas faktiskās galvenes
header_rows = None
header_rows

In [ ]:
# TODO: sadaliet failu blokos pēc galveņu rindām
# un apvienojiet tos vienā DataFrame
mixed = None
mixed.head()

In [ ]:
# TODO: iztīriet kolonnas un vērtības
# Ieteikumi:
# - standartizējiet kolonnu nosaukumus
# - Amount -> Amount Clean
# - Date -> Date Parsed
# - Approved -> True/False
# - noņemiet tukšās rindas un dublikātus

mixed_clean = None
mixed_clean.head()

In [ ]:
# TODO: izveidojiet kopsavilkumu pēc Municipality un Department
mixed_summary = None
mixed_summary

In [ ]:
# TODO: eksportējiet iztīrīto failu uz Excel

# 7. Reporting template — rezultātu ierakstīšana šablonā

## Uzdevums
Failā `6_reporting_template/report_template.xlsx` ir vienkāršs atskaites šablons.

### Jums jāizdara:
1. Jāizveido šablona kopija.
2. Jāatver tā ar `openpyxl`.
3. Jāieraksta galvenes dati:
   - iestāde,
   - periods.
4. Jāievieto kopsavilkuma tabula no iepriekšējā uzdevuma vai no budžeta datiem.
5. Jāsaglabā rezultāts.

In [ ]:
template_file = DATA_ROOT / "6_reporting_template" / "report_template.xlsx"
report_output = OUTPUT_DIR / "student_report_from_template.xlsx"

# TODO: izveidojiet šablona kopiju

In [ ]:
# TODO: nolasiet workbook ar openpyxl
# un aizpildiet šūnas ar virsraksta informāciju

In [ ]:
# TODO: ierakstiet tabulas datus, sākot no noteiktas rindas
# Piemēram, izmantojiet report_data no budžeta kopsavilkuma

In [ ]:
# TODO: saglabājiet workbook

# 8. Chart data — diagrammas ievietošana ar `openpyxl`

## Uzdevums
Failā `7_chart_data/monthly_summary.xlsx` ir mēnešu kopsavilkuma tabula.

### Jums jāizdara:
1. Jāatver fails ar `openpyxl`.
2. Jāizveido `BarChart`.
3. Jānorāda datu diapazons un kategoriju diapazons.
4. Jāievieto diagramma darba lapā.
5. Jāsaglabā rezultāts jaunā failā.

In [ ]:
chart_file = DATA_ROOT / "7_chart_data" / "monthly_summary.xlsx"
chart_output = OUTPUT_DIR / "student_chart_data.xlsx"

# TODO: izveidojiet faila kopiju un atveriet to ar openpyxl

In [ ]:
# TODO: izveidojiet BarChart objektu
# Padomi:
# - chart.title
# - chart.y_axis.title
# - chart.x_axis.title

In [ ]:
# TODO: norādiet data un cats ar Reference(...)
# un pievienojiet diagrammu lapai

In [ ]:
# TODO: saglabājiet workbook

# 9. Papildu refleksijas jautājumi

Atbildiet īsi rakstiski vai pārrunājiet grupā:

1. Kurš uzdevums visvairāk atgādināja reālu darba situāciju?
2. Kur bija lielākā atšķirība starp “Excel domāšanu” un “Python domāšanu”?
3. Kurā vietā `pandas` bija ērtāks nekā `openpyxl`?
4. Kur automatizācija visvairāk samazina kļūdu iespējamību?
5. Kā jūs šo pieeju varētu pielāgot savam darbam?

# 10. Noslēgums

Kad pabeidzat šo workbook:
- jums jābūt vairākām jaunām Excel izejām mapē `student_outputs`,
- jums jāspēj izskaidrot galvenos apstrādes soļus,
- un jums jāredz, kā Python var pārņemt atkārtojošus Excel darbus.

Šī darba grāmata ir saistīta ar kursa 5. tēmu par Python izmantošanu Excel dokumentu apstrādē, kas atbilst kursa kopējai 8 lekciju struktūrai. fileciteturn1file0